# 第10章：Xyce 线性求解器 Adapter 原型部署

> 实验主入口：[Xyce 线性求解器 Adapter 原型部署实验手册](EXPERIMENT_GUIDE.md)。只有实际启动 upstream Xyce 时才可称为完整 Xyce 仿真。

本章把前面学习的 GMRES/SpMV 后端放入应用 wrapper。原 `Ascend-Xyce` 工程不修改 Xyce 核心源码，而用 adapter 模拟 Xyce 的 `solve(A,b,x)` 线性求解接口，复用 `Ascend-GMRES`，完成依赖准备、构建、启动、CSV 结果管理和故障定位。

## 实验条件

默认可复现对象是 `xyce_benchmark` wrapper，不是读取 netlist 的完整 upstream Xyce 仿真，也不包含 Slurm/PBS/LSF。可选脚本能拉取或尝试构建上游 Xyce，但默认教学闭环不依赖网络。课程所说“部署”指应用依赖和 backend 的可复现部署运行。

## 学习目标

- 解释 Xyce wrapper → adapter → Ascend-GMRES 的依赖关系
- 使用 vendored dependency 完成离线构建
- 管理矩阵输入、运行参数、CSV 和退出状态
- 区分 wrapper 总时间、assembly 和 linear solver 时间

## 环境检查

直接检查 Ascend NPU、CANN、acl_rtc 与 CMake 环境；若失败，先加载目标节点的 CANN 工具链。


In [ ]:
import platform, shutil
print("Python:", platform.python_version())
print("CMake:", shutil.which("cmake"))
print("说明：本章默认运行 Xyce Adapter benchmark，不宣称完整 upstream Xyce 仿真。")


## 章节内容

- [10.01_chapter_intro](10.01_chapter_intro.ipynb)：应用部署定位
- [10.02_xyce_application_and_adapter](10.02_xyce_application_and_adapter.ipynb)：wrapper/adapter 调用链
- [10.03_dependency_and_build_deployment](10.03_dependency_and_build_deployment.ipynb)：依赖解析与构建
- [10.04_benchmark_launch_and_configuration](10.04_benchmark_launch_and_configuration.ipynb)：启动和输入配置
- [10.05_logs_results_and_troubleshooting](10.05_logs_results_and_troubleshooting.ipynb)：CSV、结果和排障
- [10.06_chapter_test](10.06_chapter_test.ipynb)：可复现部署实践

## 实验工程说明与本章任务

本章实验工程位于 `src/ascend_xyce/`。`xyce_adapter.cpp`/`.hpp` 实现 `solve(A,b,x)`；vendored Ascend-GMRES 是 solver；`xyce_benchmark.cpp` 驱动输入；脚本准备依赖、构建、运行；`results/` 保存 CSV。

### 本章实验任务

检查 vendored backend → 离线构建 → 配置 warmup/repeat/matrix → 调用 solve → 检查收敛/残差/误差 → 读取 CSV/log → 分层排错。

所有路径均相对 Notebook 当前目录。先检查环境，再运行真实工程；历史结果只用于观察趋势。

### 完成标准

能够指出核心源码和脚本职责，完成可用环境内的构建/诊断，并按“Matrix、Solver、Simulation/Linear time、Iterations、Residual、Converged、Error、日志”记录证据。


## 实验记录与练习

先确认 Actual backend、Device、退出码与正确性，再记录当前输出。历史 HostPrototype CSV 不能作为本次 NPU 实测。

完成后回答：实际后端是什么？reference 与 tolerance 是什么？主要耗时来自计算、通信、传输还是同步？改变一个并发或算法参数后，正确性和性能如何变化？参考答案仅通过本章 `answer/` 链接查阅。
